In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Carregar os dados
# O arquivo S022.csv contém as leituras dos sensores e os rótulos de atividade (label)
try:
    df = pd.read_csv('S022.csv')
    print("Dados carregados com sucesso!")
except FileNotFoundError:
    # Caso o arquivo não esteja no diretório raiz, tenta carregar simulando ou alertando
    print("Arquivo S022.csv não encontrado. Por favor, faça o upload do arquivo para o ambiente do Colab.")
    # Criando dados fictícios apenas para evitar que o código quebre caso falte o arquivo no teste inicial
    columns = ['timestamp', 'back_x', 'back_y', 'back_z', 'thigh_x', 'thigh_y', 'thigh_z', 'label']
    df = pd.DataFrame(np.random.randn(1000, 8), columns=columns)
    df['label'] = np.random.choice([1, 2, 3, 4, 7, 8, 13, 14], size=1000) # Rótulos típicos do HARTH

# 2. Mapeamento e Preparação dos Dados
# Com base no HARTH (Human Activity Recognition Trondheim Home):
# Comportamento Sedentário (Sedentary): Walking/Running/Cycling geralmente são alta intensidade.
# Vamos mapear os códigos numéricos padrão do HARTH para nossas 3 classes de interesse:
# 1: walking, 2: running, 3: shuffling, 4: stairs (going up), 5: stairs (going down),
# 7: standing, 8: sitting, 13: cycling (sitting), 14: cycling (standing), 130: cycling (sitting, leisure), 140: cycling (standing, leisure)

def categorizar_atividade(label):
    # Sedentário: sitting (8), lying (normalmente mapeado ou outro código deitado)
    if label in [8, 9]:
        return 'Sedentário (Sitting/Lying)'
    # Atividade Leve: standing (7), shuffling (3)
    elif label in [3, 7]:
        return 'Atividade Leve (Standing/Shuffling)'
    # Alta Intensidade: walking (1), running (2), cycling (13, 14, 130, 140), stairs (4, 5)
    elif label in [1, 2, 4, 5, 13, 14, 130, 140]:
        return 'Alta Intensidade (Walking/Running/Cycling)'
    else:
        return 'Outros'

# Aplicar o mapeamento
df['categoria'] = df['label'].apply(categorizar_atividade)

# Remover classes não identificadas ou 'Outros' se houver
df = df[df['categoria'] != 'Outros']

# 3. Definição de Features e Alvo (Target)
# Utilizaremos as leituras dos acelerômetros da lombar (back) e da coxa (thigh) como features
features = ['back_x', 'back_y', 'back_z', 'thigh_x', 'thigh_y', 'thigh_z']
X = df[features]
y = df['categoria']

# 4. Divisão em Treino e Teste
# Divisão clássica de 80% para treino e 20% para teste com semente aleatória 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Treinamento do Modelo
# Usamos o RandomForestClassifier, um modelo robusto baseado em árvores de decisão para classificação
modelo = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
modelo.fit(X_train, y_train)

# 6. Predição e Avaliação
y_pred = modelo.predict(X_test)

# Exibir a distribuição do tempo que o usuário passa em cada estado (com base no dataset completo)
print("\n--- DISTRIBUIÇÃO DO TEMPO POR CATEGORIA (Sessão Completa) ---")
total_registros = len(df)
for cat, count in df['categoria'].value_counts().items():
    # Cada registro equivale a 0.02 segundos (50 Hz)
    segundos = count * 0.02
    minutos = segundos / 60
    porcentagem = (count / total_registros) * 100
    print(f"{cat}: {minutos:.2f} minutos ({porcentagem:.1f} % dos dados)")

print("\n--- RELATÓRIO DE MÉTRICAS DO MODELO ---")
print(f"Acurácia Geral: {accuracy_score(y_test, y_pred):.4f}")
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred))

In [3]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score
from google.colab import files

# 1. Detectar o tipo de problema de forma automática
# Verificamos o tipo de dado da variável de saída 'y'
alvo_tipo = y.dtype
se_classificacao = isinstance(alvo_tipo, object) or pd.api.types.is_categorical_dtype(alvo_tipo)

# 2. Estruturar os dados base para a plataforma de visualização
resultado = {
    "base": "S*.csv (Grupo de Participantes)",
    "alvo": "categoria",
    "tipo": "classificacao" if se_classificacao else "regressao",
    "entradas": list(features)
}

if se_classificacao:
    # Calcular acurácia global do modelo no conjunto de teste
    acuracia = float(accuracy_score(y_test, y_pred))

    # Calcular o piso (proporção da classe mais comum em todos os dados utilizados no treino/teste)
    classe_mais_comum = y.value_counts().idxmax()
    piso = float(y.value_counts().max() / len(y))

    # Obter lista de classes únicas de forma consistente e ordenada
    classes_unicas = sorted(list(y_test.unique()))

    # Gerar a matriz de confusão alinhada com a ordem das classes
    matriz = confusion_matrix(y_test, y_pred, labels=classes_unicas)
    matriz_confusao = matriz.tolist()

    # Extrair até 100 amostras do conjunto de teste para a visualização
    amostras_df = pd.DataFrame({"real": y_test.astype(str), "previsto": y_pred.astype(str)}).head(100)
    amostras = amostras_df.to_dict(orient="records")

    # Alimentar o dicionário com os dados obtidos
    resultado["acuracia"] = acuracia
    resultado["piso"] = piso
    resultado["classes"] = classes_unicas
    resultado["matriz_confusao"] = matriz_confusao
    resultado["amostras"] = amostras

else:
    # Implementação preventiva para o caso de modelos de regressão
    from sklearn.metrics import mean_absolute_error
    mae_modelo = float(mean_absolute_error(y_test, y_pred))
    previsao_piso = np.full_like(y_test, y_train.mean(), dtype=float)
    mae_piso = float(mean_absolute_error(y_test, previsao_piso))

    amostras_df = pd.DataFrame({"real": y_test, "previsto": y_pred}).head(100)
    amostras = amostras_df.to_dict(orient="records")

    resultado["erro_medio_modelo"] = mae_modelo
    resultado["erro_medio_piso"] = mae_piso
    resultado["amostras"] = amostras

# 3. Salvar o dicionário como arquivo JSON mantendo caracteres especiais
nome_arquivo = "resultado_projeto.json"
with open(nome_arquivo, "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=4)

print(f"Arquivo '{nome_arquivo}' gerado com sucesso!")

# 4. Baixar o arquivo gerado para a máquina local do usuário
files.download(nome_arquivo)

Arquivo 'resultado_projeto.json' gerado com sucesso!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import pandas as pd
import numpy as np
import glob
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Carregar de forma otimizada os dados de todos os participantes (S006 a S029)
caminhos_arquivos = glob.glob('/content/S*.csv')

if len(caminhos_arquivos) == 0:
    print("Nenhum arquivo de participante (S*.csv) foi encontrado. Usando dados fictícios agrupados para demonstração.")
    caminhos_arquivos = ['S022.csv']

# Lendo apenas as colunas necessárias para economizar muita memória e acelerar o processo
colunas_interesse = ['back_x', 'back_y', 'back_z', 'thigh_x', 'thigh_y', 'thigh_z', 'label']
lista_dfs = []

for caminho in caminhos_arquivos:
    try:
        df_temp = pd.read_csv(caminho, usecols=colunas_interesse)
        lista_dfs.append(df_temp)
    except Exception as e:
        print(f"Erro ao ler o arquivo {caminho}: {e}")

# Concatenar todos os datasets em um único DataFrame global
if lista_dfs:
    df_grupo = pd.concat(lista_dfs, ignore_index=True)
    print(f"Dados carregados com sucesso! Total de registros do grupo original: {len(df_grupo)}")
else:
    raise FileNotFoundError("Nenhum arquivo CSV pôde ser carregado.")

# 2. Mapeamento e Categorização das Atividades
def categorizar_atividade(label):
    # Sedentário: sitting (8), lying (9)
    if label in [8, 9]:
        return 'Sedentário (Sitting/Lying)'
    # Atividade Leve: standing (7), shuffling (3)
    elif label in [3, 7]:
        return 'Atividade Leve (Standing/Shuffling)'
    # Alta Intensidade: walking (1), running (2), cycling (13, 14, 130, 140), stairs (4, 5)
    elif label in [1, 2, 4, 5, 13, 14, 130, 140]:
        return 'Alta Intensidade (Walking/Running/Cycling)'
    else:
        return 'Outros'

# Aplicar o mapeamento de categorias
df_grupo['categoria'] = df_grupo['label'].apply(categorizar_atividade)

# Remover registros classificados como 'Outros'
df_grupo = df_grupo[df_grupo['categoria'] != 'Outros']

# 3. Análise de Tempo do Grupo (Proporção Média de Tempo Real)
print("\n--- PROPORÇÃO MÉDIA DE TEMPO REAL DO GRUPO (Dados Completos) ---")
total_registros_grupo = len(df_grupo)
for cat, count in df_grupo['categoria'].value_counts().items():
    # Cada amostragem representa 0.02 segundos (50 Hz)
    segundos_total = count * 0.02
    minutos_total = segundos_total / 60
    porcentagem_total = (count / total_registros_grupo) * 100
    print(f"{cat}: {minutos_total:.2f} minutos no total ({porcentagem_total:.2f}% do tempo do grupo)")

# 4. Amostragem Estratificada para Treinamento Rápido (Evitar travamento)
# Amostramos apenas 100.000 linhas preservando a proporção exata das categorias
df_amostra = df_grupo.groupby('categoria', group_keys=False).apply(lambda x: x.sample(min(len(x), 33000), random_state=42))
print(f"\nTamanho da amostra otimizada para o treinamento: {len(df_amostra)} registros.")

# 5. Definição de Features e Variável Alvo
features = ['back_x', 'back_y', 'back_z', 'thigh_x', 'thigh_y', 'thigh_z']
X = df_amostra[features]
y = df_amostra['categoria']

# 6. Divisão em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 7. Treinamento Otimizado do Modelo
# Reduzimos para 50 estimadores para um processamento quase instantâneo
modelo_grupo = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, max_depth=15)
modelo_grupo.fit(X_train, y_train)

# 8. Avaliação rápida
y_pred = modelo_grupo.predict(X_test)

print("\n--- RELATÓRIO DE MÉTRICAS DO MODELO (DADOS DO GRUPO) ---")
print(f"Acurácia Geral: {accuracy_score(y_test, y_pred):.4f}")
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred))

Dados carregados com sucesso! Total de registros do grupo original: 6461328

--- PROPORÇÃO MÉDIA DE TEMPO REAL DO GRUPO (Dados Completos) ---
Atividade Leve (Standing/Shuffling): 1052.51 minutos no total (55.22% do tempo do grupo)
Alta Intensidade (Walking/Running/Cycling): 710.47 minutos no total (37.28% do tempo do grupo)
Sedentário (Sitting/Lying): 142.97 minutos no total (7.50% do tempo do grupo)


/tmp/ipykernel_11844/1724100664.py:66: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_amostra = df_grupo.groupby('categoria', group_keys=False).apply(lambda x: x.sample(min(len(x), 33000), random_state=42))



Tamanho da amostra otimizada para o treinamento: 99000 registros.

--- RELATÓRIO DE MÉTRICAS DO MODELO (DADOS DO GRUPO) ---
Acurácia Geral: 0.9719

Relatório de Classificação Detalhado:
                                            precision    recall  f1-score   support

Alta Intensidade (Walking/Running/Cycling)       0.93      0.99      0.96      6600
       Atividade Leve (Standing/Shuffling)       0.99      0.93      0.96      6600
                Sedentário (Sitting/Lying)       1.00      1.00      1.00      6600

                                  accuracy                           0.97     19800
                                 macro avg       0.97      0.97      0.97     19800
                              weighted avg       0.97      0.97      0.97     19800

